In [ ]:
import torch.nn as nn
import torch.functional as F

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-06-10 09:26:36--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.06s   

2026-06-10 09:26:36 (19.0 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
with open('input.txt','r') as f:
  text=f.read()
  print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [ ]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(''.join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
stoi={s:i for i,s in enumerate(chars)}
itos={i:s for s,i in stoi.items()}
encode=lambda ch :[stoi[s] for s in ch]
decode=lambda num:''.join([itos[i] for i in num ])


In [ ]:
import torch
data=torch.tensor(encode(text))
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]



In [ ]:
batch_size=32
block_size=32

def get_batch(split):
  data=train_data if split=='train' else val_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y



x,y =get_batch('train')
for b in range (batch_size):
  for t in range(block_size):
    context=x[b,:t+1]
    target=y[b,t]
    print(f'for{context} target is {target}')








fortensor([58]) target is 46
fortensor([58, 46]) target is 39
fortensor([58, 46, 39]) target is 52
fortensor([58, 46, 39, 52]) target is 49
fortensor([58, 46, 39, 52, 49]) target is 44
fortensor([58, 46, 39, 52, 49, 44]) target is 59
fortensor([58, 46, 39, 52, 49, 44, 59]) target is 50
fortensor([58, 46, 39, 52, 49, 44, 59, 50]) target is 1
fortensor([58, 46, 39, 52, 49, 44, 59, 50,  1]) target is 43
fortensor([58, 46, 39, 52, 49, 44, 59, 50,  1, 43]) target is 60
fortensor([58, 46, 39, 52, 49, 44, 59, 50,  1, 43, 60]) target is 43
fortensor([58, 46, 39, 52, 49, 44, 59, 50,  1, 43, 60, 43]) target is 52
fortensor([58, 46, 39, 52, 49, 44, 59, 50,  1, 43, 60, 43, 52]) target is 1
fortensor([58, 46, 39, 52, 49, 44, 59, 50,  1, 43, 60, 43, 52,  1]) target is 44
fortensor([58, 46, 39, 52, 49, 44, 59, 50,  1, 43, 60, 43, 52,  1, 44]) target is 53
fortensor([58, 46, 39, 52, 49, 44, 59, 50,  1, 43, 60, 43, 52,  1, 44, 53]) target is 56
fortensor([58, 46, 39, 52, 49, 44, 59, 50,  1, 43, 60, 43,

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)
#in--(B,T)----> (B,T,C),c:vocab_size

class BigramLanguageModel(nn.Module):

    def __init__(self,vocab_size=65,n_embedd=32):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,n_embedd)
        self.lm_head=nn.Linear(n_embedd,vocab_size)


    def forward(self,idx,target=None):
        tok_emb=self.token_embedding_table(idx) #-->(B,T,C)
        logits=self.lm_head(tok_emb)
        B,T,C=logits.shape
        if target is None:
            loss=None

        else:
                logits=logits.view(B*T,C)
                target=target.view(B*T)
                loss=F.cross_entropy(logits,target)
        return logits,loss


    def generate(self,idx,token_max):
        for i in range(token_max):
            logits,loss=self.forward(idx)
            logits=logits[:,-1,:] # last column-->(B,C)
            probs=torch.softmaxt(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx




In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)
#in--(B,T)----> (B,T,C),c:vocab_size
block_size=100
head_size=32
n_embedd=32
vocab_size=65

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,n_embedd)
        self.pos_embedding_table=nn.Embedding(block_size,n_embedd)
        self.head=head(head_size)
        self.lm_head=nn.Linear(head_size,vocab_size)


    def forward(self,idx,target=None):
        B,T=idx.shape
        tok_emb=self.token_embedding_table(idx) #-->(B,T,C)
        pos_emb=self.pos_embedding_table((torch.arange(T))) #-->(T,C)
        x=tok_emb+pos_emb #(B,T,C)
        x=self.head(x)
        logits=self.lm_head(x)
        B,T,C=logits.shape
        if target is None:
            loss=None

        else:
                logits=logits.view(B*T,C)
                target=target.view(B*T)
                loss=F.cross_entropy(logits,target)
        return logits,loss


    def generate(self,idx,token_max):
        for i in range(token_max):
            idx_cond=idx[:,-block_size:]
            logits,loss=self.forward(idx_cond)
            logits=logits[:,-1,:] # last column-->(B,C)
            probs=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx


class head(nn.Module):
     def __init__(self,head_size):
        super().__init__()
        self.key=nn.Linear(n_embedd,head_size)
        self.query=nn.Linear(n_embedd,head_size)
        self.value=nn.Linear(n_embedd,head_size)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
     def forward(self,x): #x-->(B,T,n_embedd)
         B,T,C=x.shape
         k=self.key(x) #k-->(B,T,head_size)
         q=self.query(x)
         v=self.value(x)

         wei=q @ k.transpose(-2,-1) *C**-0.5 #-->(B,T,T)=(B,n_embedd,n_embedd)
         wei=wei.masked_fill(self.tril[:T,:T]==0,float('-inf'))
         wei=torch.softmax(wei,dim=-1)
         out=wei @ v #(B,T,T) @ (B,T,head_size)-->(B,T,head_size )

         return out


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)
#in--(B,T)----> (B,T,C),c:vocab_size
block_size=50
n_embedd=256
vocab_size=65
n_head=4

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,n_embedd)
        self.pos_embedding_table=nn.Embedding(block_size,n_embedd)
        self.heads=MultiHeadAttention(n_head,head_size)
        self.lm_head=nn.Linear(n_embedd,vocab_size)


    def forward(self,idx,target=None):
        B,T=idx.shape
        tok_emb=self.token_embedding_table(idx) #-->(B,T,C)
        pos_emb=self.pos_embedding_table((torch.arange(T))) #-->(T,C)
        x=tok_emb+pos_emb #(B,T,C)
        x=self.heads(x)
        logits=self.lm_head(x)
        B,T,C=logits.shape
        if target is None:
            loss=None

        else:
                logits=logits.view(B*T,C)
                target=target.view(B*T)
                loss=F.cross_entropy(logits,target)
        return logits,loss


    def generate(self,idx,token_max):
        for i in range(token_max):
            idx_cond=idx[:,-block_size:]
            logits,loss=self.forward(idx_cond)
            logits=logits[:,-1,:] # last column-->(B,C)
            probs=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx


class head(nn.Module):
     def __init__(self,head_size):
        super().__init__()
        self.key=nn.Linear(n_embedd,head_size)
        self.query=nn.Linear(n_embedd,head_size)
        self.value=nn.Linear(n_embedd,head_size)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
     def forward(self,x): #x-->(B,T,n_embedd)
         B,T,C=x.shape
         k=self.key(x) #k-->(B,T,head_size)
         q=self.query(x)
         v=self.value(x)

         wei=q @ k.transpose(-2,-1) *C**-0.5 #-->(B,T,T)=(B,n_embedd,n_embedd)
         wei=wei.masked_fill(self.tril[:T,:T]==0,float('-inf'))
         wei=torch.softmax(wei,dim=-1)
         out=wei @ v #(B,T,T) @ (B,T,head_size)-->(B,T,head_size )

         return out

class MultiHeadAttention(nn.Module):
    def __init__(self,n_head,head_size):
        super().__init__()
        head_size=n_embedd//n_head
        self.heads=nn.ModuleList([head(head_size) for i in range(n_head)])
    def forward(self,x):
        out=torch.cat([h(x) for h in self.heads],dim=-1)
        return out

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)
#in--(B,T)----> (B,T,C),c:vocab_size
block_size=64
n_embedd=128
vocab_size=65
n_head=4
dropout=0.2

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,n_embedd)
        self.pos_embedding_table=nn.Embedding(block_size,n_embedd)
        self.blocks=nn.Sequential(
            block(n_embedd,n_head=4),
            block(n_embedd,n_head=4),
            block(n_embedd,n_head=4),
            block(n_embedd,n_head=4),
            nn.LayerNorm(n_embedd),
        )
        self.lm_head=nn.Linear(n_embedd,vocab_size)


    def forward(self,idx,target=None):
        B,T=idx.shape
        tok_emb=self.token_embedding_table(idx) #-->(B,T,C)
        pos_emb=self.pos_embedding_table((torch.arange(T))) #-->(T,C)
        x=tok_emb+pos_emb #(B,T,C)
        x=self.blocks(x)
        logits=self.lm_head(x)
        B,T,C=logits.shape
        if target is None:
            loss=None

        else:
                logits=logits.view(B*T,C)
                target=target.view(B*T)
                loss=F.cross_entropy(logits,target)
        return logits,loss


    def generate(self,idx,token_max):
        for i in range(token_max):
            idx_cond=idx[:,-block_size:]
            logits,loss=self.forward(idx_cond)
            logits=logits[:,-1,:] # last column-->(B,C)
            probs=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx


class head(nn.Module):
     def __init__(self,head_size):
        super().__init__()
        self.key=nn.Linear(n_embedd,head_size)
        self.query=nn.Linear(n_embedd,head_size)
        self.value=nn.Linear(n_embedd,head_size)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
        self.dropout=nn.Dropout(dropout)
     def forward(self,x): #x-->(B,T,n_embedd)
         B,T,C=x.shape
         k=self.key(x) #k-->(B,T,head_size)
         q=self.query(x)
         v=self.value(x)

         wei=q @ k.transpose(-2,-1) *C**-0.5 #-->(B,T,T)=(B,n_embedd,n_embedd)
         wei=wei.masked_fill(self.tril[:T,:T]==0,float('-inf'))
         wei=torch.softmax(wei,dim=-1)
         wei=self.dropout(wei)
         out=wei @ v #(B,T,T) @ (B,T,head_size)-->(B,T,head_size )

         return out

class MultiHeadAttention(nn.Module):
    def __init__(self,n_head,head_size):
        super().__init__()
        self.heads=nn.ModuleList([head(head_size) for i in range(n_head)])
        self.proj=nn.Linear(n_embedd,n_embedd)
    def forward(self,x):
        out=torch.cat([h(x) for h in self.heads],dim=-1)
        out=self.proj(out)
        return out

class Feedforward(nn.Module):
    def __init__(self,n_embedd):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(n_embedd,4*n_embedd),
            nn.ReLU(),
            nn.Linear(4*n_embedd,n_embedd),
            nn.Dropout(dropout)

        )

    def forward(self,x):
        return self.net(x)


class block(nn.Module):
    def __init__(self,n_embedd,n_head):
        super().__init__()
        head_size=n_embedd//n_head
        self.heads=MultiHeadAttention(n_head,head_size)
        self.ffwd=Feedforward(n_embedd)
        self.ln1=nn.LayerNorm(n_embedd)
        self.ln2=nn.LayerNorm(n_embedd)

    def forward(self,x):
        x=x+self.heads(self.ln1(x))
        x=x+self.ffwd(self.ln2(x))
        return x


In [ ]:
m=BigramLanguageModel()

In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-4)

In [ ]:
m = BigramLanguageModel()
optimizer = torch.optim.AdamW(m.parameters(), lr=3e-4)

@torch.no_grad()
def estimate_loss(steps=200):
    m.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(steps)
        for k in range(steps):
            xb, yb = get_batch(split)
            logits, loss = m(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

for i in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 500 == 0:
        losses = estimate_loss()
        print(f"step {i}: train {losses['train']:.4f} | val {losses['val']:.4f}")

step 0: train 4.1532 | val 4.1591
step 500: train 2.3512 | val 2.3634
step 1000: train 2.1063 | val 2.1334
step 1500: train 1.9534 | val 2.0146
step 2000: train 1.8386 | val 1.9335
step 2500: train 1.7606 | val 1.8859
step 3000: train 1.7085 | val 1.8458
step 3500: train 1.6586 | val 1.8181
step 4000: train 1.6252 | val 1.7883
step 4500: train 1.5988 | val 1.7659
step 5000: train 1.5740 | val 1.7396
step 5500: train 1.5482 | val 1.7289
step 6000: train 1.5325 | val 1.7162
step 6500: train 1.5041 | val 1.6922
step 7000: train 1.4981 | val 1.6873
step 7500: train 1.4812 | val 1.6781
step 8000: train 1.4748 | val 1.6677
step 8500: train 1.4592 | val 1.6518
step 9000: train 1.4537 | val 1.6515
step 9500: train 1.4447 | val 1.6418


In [ ]:
m.eval()
context = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(context, token_max=1000)[0].tolist()))




Second Senator:
That he lighty thus what he
Be the your eaptions
Farewed with thee say a day to my tress'd endier
point what a vouchday its as the mains
Fward your station of mine tepute, earning tooe,
In traitorst thou chry give for ourselves your
But no which thus rememberal to o' taken,
To the juster did stand what I would a gods
Warwich most of this set, is he present. The Murderer:
Thou buld shedst that come, for Might charged,
Infore them moved, insulf, I do your in that kindress:
I delight and
Which buy says she sligh the peace; you, flight,
The sorry bready of the wat weaus bloody.
Whits us, has intended out the like budge!

DUKE VINCENTIO:
I to come son.

PISABELLA:
Master base pigntate dopt.

DUKE VINCENTIO:
My lord, you repot for 'tis all, and place
With in without I'll before the grief;' now majest
The defrence power such at sleep, and fight!
Undert, now therefore, Lord's my sorrow, and thou dead:
That believes?

WARWICK:
Peter 'What may usurpphy lord,
They can serving ar

In [ ]:

for i in range(5000):
  xb,yb=get_batch('train')
  logits,loss=m(xb,yb)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
print(f'{loss=}')



NameError: name 'get_batch' is not defined

In [ ]:



id=torch.ones((10,1),dtype=torch.long)
idx=(m.generate(id,200)[0]).tolist()
out=decode(idx)




out

" fzfPAd'BQePGNNsEpDPaJEMJkMbKgIc,iSPSXbW&&UbP-OmvRXYtAbq-yN;BGV:XQLnVmvOL?-jRQemyTliqa-sI'FN.H'q:Bv\nlDDo!hSETrYhvy,voAkVXcf\n3BQtj?l3zE:jHzdT&nf&vYbWAIS,&ZWu$.qb&KAT:BQUalDIKpGwivMYvaeVvJfPbQ,DtW'UzdBcC"

In [ ]:
id=torch.ones((1,1),dtype=torch.long)
idx=(m.generate(id,200)[0]).tolist()
out=decode(idx)


NameError: name 'torch' is not defined

In [ ]:
out

" kemerowat; s s.\nHFGB-d.\nO:\n\n\nWeilamaisto t ap,\nEDelvendimm bak'They handd whe th LY\nI,\nCI RONTICUKEAnsalononghasprineriso ean.\nBOLELLLAn, w wespuepld asthe te th'\nEf andPk, lfircs kiny:\nWeath's h oind"

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)


  def forward(self,idx,target=None):#idx(B,T)*
      logits=self.token_embedding_table(idx) #(B,T,C)
      if target is None:
        loss=None
      else:
        B,T,C=logits.shape
        logits=logits.view(B*T,C)
        target=target.view(B*T)
        loss=F.cross_entropy(logits,target)
      return logits,loss


  def generate(self,idx,max_tokens):
    for _ in range(max_tokens):
      logits,loss=self(idx)
      logits=logits[:,-1,:]
      probs=torch.softmax(logits,dim=-1)
      idx_next=torch.multinomial(probs,num_samples=1)
      idx=torch.cat((idx,idx_next),dim=1)
    return idx




In [ ]:
import torch
X=torch.randn((4,8,65))


In [ ]:
tril=torch.tril